# 02 — Task 2.1.1: Lexicon & Rule-based Models
Applies **TextBlob**, **VADER**, and **Stanza** to the test set and compares performance.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from tqdm import tqdm

from src.utils import load_data, evaluate_predictions, save_results

_, test_labels = load_data('test')
test_texts, _ = load_data('test')
print(f'Test set: {len(test_texts):,} reviews')

## 1. TextBlob
Uses `polarity > 0` → `pos`, else → `neg`.

In [ ]:
from textblob import TextBlob

def textblob_predict(texts):
    preds = []
    for text in tqdm(texts, desc='TextBlob'):
        polarity = TextBlob(text).sentiment.polarity
        preds.append('pos' if polarity > 0 else 'neg')
    return preds

tb_preds = textblob_predict(test_texts)
tb_metrics = evaluate_predictions(test_labels, tb_preds)
print('TextBlob:', tb_metrics)
save_results('2.1.1', 'TextBlob', tb_metrics,
             preprocessing='raw text, polarity > 0 → pos')

## 2. VADER
Uses `compound ≥ 0.05` → `pos`, else → `neg`.

In [ ]:
import nltk
nltk.download('vader_lexicon', quiet=True)
from nltk.sentiment.vader import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

def vader_predict(texts):
    preds = []
    for text in tqdm(texts, desc='VADER'):
        compound = sia.polarity_scores(text)['compound']
        preds.append('pos' if compound >= 0.05 else 'neg')
    return preds

vader_preds = vader_predict(test_texts)
vader_metrics = evaluate_predictions(test_labels, vader_preds)
print('VADER:', vader_metrics)
save_results('2.1.1', 'VADER', vader_metrics,
             preprocessing='raw text, compound >= 0.05 → pos',
             notes='Designed for short text; long IMDB reviews may reduce accuracy')

## 3. Stanza
Uses built-in sentiment pipeline (0=neg, 1=neutral, 2=pos). Neutral mapped to neg for binary task.

In [ ]:
import stanza
# Download English model with sentiment if not already present
stanza.download('en', processors='tokenize,sentiment', verbose=False)

nlp = stanza.Pipeline('en', processors='tokenize,sentiment', verbose=False)

def stanza_predict(texts, batch_size=64):
    """Run Stanza sentiment in batches. Returns document-level label."""
    preds = []
    for i in tqdm(range(0, len(texts), batch_size), desc='Stanza'):
        batch = texts[i:i+batch_size]
        docs = [stanza.Document([], text=t) for t in batch]
        docs = nlp(docs)
        for doc in docs:
            # Average sentiment score across sentences: 0=neg,1=neutral,2=pos
            scores = [s.sentiment for s in doc.sentences]
            avg = sum(scores) / len(scores) if scores else 1
            preds.append('pos' if avg > 1 else 'neg')
    return preds

stanza_preds = stanza_predict(test_texts)
stanza_metrics = evaluate_predictions(test_labels, stanza_preds)
print('Stanza:', stanza_metrics)
save_results('2.1.1', 'Stanza', stanza_metrics,
             preprocessing='sentence-level avg sentiment > 1 → pos')

## 4. Comparison

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

results = pd.DataFrame([
    {'Tool': 'TextBlob', **tb_metrics},
    {'Tool': 'VADER',    **vader_metrics},
    {'Tool': 'Stanza',   **stanza_metrics},
])
display(results.set_index('Tool'))

ax = results.set_index('Tool')[['accuracy', 'precision', 'recall', 'f1']].plot(
    kind='bar', figsize=(9, 4), rot=0, colormap='tab10')
ax.set_ylim(0, 1)
ax.set_title('Task 2.1.1 — Lexicon/Rule-based Models on IMDB Test Set')
ax.set_ylabel('Score')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../results/fig_lexicon_rules.png', dpi=150, bbox_inches='tight')
plt.show()